# Can we predict next-month churn?
Target: `churn_next_month` (1 = account cancelled in the month after the snapshot). One row per account-month.

Plan: inspect the data, check for leakage, build a time-honest evaluation, compare a simple baseline with a gradient-boosted model, and state what the results do and don't support.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance
pd.set_option('display.width', 140)
df = pd.read_csv('data.csv')
print(df.shape, '| accounts:', df.account_ref.nunique(), '| missing:', int(df.isna().sum().sum()), '| duplicate rows:', df.duplicated().sum())
df.head()

## 1. Target and structure

In [ ]:
print('Churn rate:', df.churn_next_month.mean().round(4))
m = df.groupby('snapshot_month').churn_next_month.agg(rate='mean', n='size')
fig, ax = plt.subplots(figsize=(10, 3.5)); m.rate.plot(ax=ax, marker='o'); ax.set_ylabel('churn rate'); ax.set_title('Churn rate by snapshot month'); plt.xticks(rotation=60); plt.show()
print(df.snapshot_month.min(), '->', df.snapshot_month.max(), '| rows per account:', df.groupby('account_ref').size().describe()[['min', '50%', 'max']].to_dict())

The churn rate is far from stable: about 3-6% in spring 2024, then over 50% in Dec 2024 / Jan 2025, then back to ~10%. A random row split would be misleading, so we evaluate on later months than we train on. Accounts also appear in many months, so rows are not independent.

## 2. Leakage check: `lifetime_months`

The dictionary says `lifetime_months` is the *total months the account was active, from the CRM export of 2026-01-05*. That is only known once the account's life is over, so it can encode the outcome. Compare it with `tenure_months` (months so far, including the snapshot month):

In [ ]:
df['life_minus_tenure'] = df.lifetime_months - df.tenure_months
print(df.groupby(df.life_minus_tenure == 0).churn_next_month.agg(['mean', 'size']))
print('AUC of (lifetime == tenure) as a "predictor":', roc_auc_score(df.churn_next_month, (df.life_minus_tenure == 0).astype(int)))

Every row where `lifetime_months == tenure_months` (the account's final month) has churn = 1, and every other row has churn = 0. The feature restates the label and would not exist at prediction time. **It is excluded from all models.** Any model that includes it would look perfect and be useless.

## 3. Legitimate features

In [ ]:
df['fee_per_seat'] = df.monthly_fee / df.seats
df['hours_per_seat'] = df.usage_hours / df.seats
num = ['seats', 'monthly_fee', 'tenure_months', 'usage_hours', 'support_tickets', 'fee_per_seat', 'hours_per_seat']
cat = ['plan']
print(df.groupby('plan')[['seats', 'monthly_fee', 'usage_hours', 'support_tickets', 'churn_next_month']].mean().round(3))
fig, axs = plt.subplots(1, 3, figsize=(12, 3))
for ax, c in zip(axs, ['tenure_months', 'usage_hours', 'support_tickets']):
    q = pd.qcut(df[c], 8, duplicates='drop') if df[c].nunique() > 8 else df[c]
    df.groupby(q, observed=True).churn_next_month.mean().plot(ax=ax, marker='o'); ax.set_title(f'churn vs {c}'); ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

## 4. Evaluation design
- **Time-based split**: train on snapshots up to 2025-04, skip 2025-05 (its label depends on June), test on 2025-06 to 2025-11.
- Metrics: ROC AUC and average precision (the base rate is the AP reference).
- Calendar month is *not* a feature: with 23 months, the Nov-Jan spike would just be memorised as a one-off.

In [ ]:
train = df[df.snapshot_month <= '2025-04']; test = df[df.snapshot_month >= '2025-06']
Xtr, ytr, Xte, yte = train[num + cat], train.churn_next_month, test[num + cat], test.churn_next_month
print(len(train), len(test), '| train rate', ytr.mean().round(3), '| test rate', yte.mean().round(3))
print('share of test accounts also seen in train:', test.account_ref.isin(train.account_ref).mean().round(3))

In [ ]:
models = {
    'logistic': make_pipeline(ColumnTransformer([('n', StandardScaler(), num), ('c', OneHotEncoder(), cat)]), LogisticRegression(max_iter=1000)),
    'grad. boosting': make_pipeline(ColumnTransformer([('n', 'passthrough', num), ('c', OrdinalEncoder(), cat)]),
                                    HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)),
}
res = {}
for k, mod in models.items():
    mod.fit(Xtr, ytr); res[k] = mod.predict_proba(Xte)[:, 1]
    print(f'{k:15s} AUC={roc_auc_score(yte, res[k]):.3f}  AP={average_precision_score(yte, res[k]):.3f}')
print(f'{"base rate":15s} AUC=0.500  AP={yte.mean():.3f}')

### Robustness: AUC by test month and an account-level bootstrap

In [ ]:
best = max(res, key=lambda k: roc_auc_score(yte, res[k])); print('best model:', best)
t = test.assign(p=res[best])
print(t.groupby('snapshot_month').apply(lambda g: pd.Series({'n': len(g), 'rate': g.churn_next_month.mean(), 'AUC': roc_auc_score(g.churn_next_month, g.p)}), include_groups=False).round(3))
rng = np.random.default_rng(0); grp = {a: g for a, g in t.groupby('account_ref')}; accts = list(grp)
aucs = [roc_auc_score(*(lambda s: (s.churn_next_month, s.p))(pd.concat([grp[a] for a in rng.choice(accts, len(accts))]))) for _ in range(200)]
print('AUC 95% CI (account bootstrap):', np.percentile(aucs, [2.5, 97.5]).round(3))

### Drivers and practical usefulness

In [ ]:
pi = permutation_importance(models[best], Xte, yte, scoring='roc_auc', n_repeats=10, random_state=0)
print(pd.Series(pi.importances_mean, index=num + cat).sort_values(ascending=False).round(4))
t = t.sort_values('p', ascending=False)
for frac in [0.05, 0.1, 0.2]:
    top = t.head(int(len(t) * frac))
    print(f'Top {frac:.0%} by risk: precision={top.churn_next_month.mean():.3f}  recall={top.churn_next_month.sum() / t.churn_next_month.sum():.3f}  lift={top.churn_next_month.mean() / t.churn_next_month.mean():.2f}x')

## 5. Conclusions
- **Leakage:** `lifetime_months` equals `tenure_months` exactly in each account's last month, which is exactly when `churn_next_month = 1`. It gives AUC 1.0, but it is only known after the fact, so it was dropped. If it was meant to be a usable feature, the CRM export needs to be redefined to be point-in-time.
- **Modest real signal:** on later, unseen months, AUC is about 0.69 (account-bootstrap 95% CI ~0.66-0.71) against 0.50 for chance. Average precision is ~0.22 vs a 9.3% base rate. The gradient-boosted model is no better than logistic regression, so the signal is simple.
- **Drivers:** support tickets (more tickets, more churn) and usage hours carry nearly all the signal. Plan, seats and fee add essentially nothing. The tenure effect is not stable.
- **Usefulness:** contacting the top 10% highest-risk account-months catches ~26% of churners at ~24% precision (2.6x lift). That is useful for prioritising outreach, not for confident individual predictions.
- **Caveats:** the churn rate swings from 3% to 55% by month (the Nov 2024 - Jan 2025 spike is unexplained by these features), so absolute probabilities will not transfer and need recalibration. Test accounts overlap with training accounts (~41%). Results come from one split of one dataset; a fresh holdout would be the real confirmation.